# Hindi UPOS Tagging: Baseline Fine-Tuning and Partial Layer Freezing

This notebook fine-tunes multilingual DistilBERT (`distilbert-base-multilingual-cased`) for Hindi Universal Part-of-Speech (UPOS) tagging using a subset of 2,500 sentences from the Hindi UD-HDTB training data. The baseline model is fine-tuned for two epochs, then compared with a partially frozen model in which the first three of six Transformer layers are frozen. Both models are evaluated using accuracy, precision, recall, and F1-score on the validation and held-out test sets.

The baseline model achieves a test F1-score of **88.19%**, compared with **86.18%** for the partially frozen model. These results suggest that, under the current training setup, full fine-tuning performs better than partial layer freezing for Hindi UPOS tagging.

In [3]:
from pathlib import Path
import sys
import json

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import torch
import transformers
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer

from src.conllu_utils import tokenize_and_align_labels, read_conllu_sentences
from src.evaluation import evaluate_token_predictions

### Environment Verification

In [4]:
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch version: 2.14.1
Transformers version: 4.57.6
MPS available: True


PyTorch 2.14.1 and Transformers 4.57.6 are installed. Apple Metal Performance Shaders (MPS) is available for GPU acceleration.

### Load Training Subset

In [5]:
data_dir = project_root / "data"
sample_path = data_dir / "processed" / "hi_hdtb_train_2500_seed42.json"

with sample_path.open("r", encoding='utf-8') as file:
    train_sample = json.load(file)

print("Number of sentences:", len(train_sample))

Number of sentences: 2500


Successfully loaded the preprocessed training subset containing 2,500 Hindi sentences, generated using seed 42.

### Load Tokenizer

In [6]:
model_checkpoint = "distilbert/distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

Load the pretrained multilingual DistilBERT tokenizer to convert Hindi text into model-compatible tokens.

### Define UPOS Labels

In [7]:
#Collect all UPOS labels from the training sample
upos_label = sorted({token["UPOS"] for sentence in train_sample
                                   for token in sentence})

label2id = {label: label_id for label_id, label in enumerate(upos_label)}

id2label = {label_id: label for label, label_id in label2id.items()}

print("Number of labels:", len(upos_label))
print("Labels:", upos_label)

Number of labels: 16
Labels: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'VERB', 'X']


The training sample contains 16 unique UPOS labels, covering categories such as nouns, verbs, adjectives, and punctuation. Each label is mapped to a unique integer ID for model training.

### Inspect Tokenization

In [8]:
example_sentence = train_sample[0]

example_tokens = [token["FORM"] for token in example_sentence]
example_labels = [token["UPOS"] for token in example_sentence]

encoded = tokenizer(example_tokens, is_split_into_words=True)

subtokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

for subtoken, word_id in zip(subtokens, encoded.word_ids()):
    print(subtoken, word_id)

[CLS] None
स 0
##्व 0
##ास 0
##्थ 0
##्य 0
क्षेत्र 1
के 2
लिए 3
बी 4
##ता 4
वर्ष 5
भ 6
##ारी 6
उ 7
##थ 7
##ल 7
- 8
प 9
##ु 9
##थ 9
##ल 9
भर 10
##ा 10
रहा 11
। 12
[SEP] None


The tokenizer splits some Hindi words into multiple subtokens and assigns each subtoken its original word index. Special tokens [CLS] and [SEP] have None as their word ID. These word IDs are used to align UPOS labels with the tokenized input.

### Align UPOS Labels with Subtokens

In [9]:
word_ids = encoded.word_ids()

aligned_labels = []
previous_word_id = None

for word_id in word_ids:
    if word_id is None or word_id == previous_word_id:
        # Special tokens such as [CLS] and [SEP]
        aligned_labels.append(-100)

    else:
        # First subtoken of a word: keep its UPOS label
        aligned_labels.append(label2id[example_labels[word_id]])

    previous_word_id = word_id

print("Number of subtokens:", len(encoded["input_ids"]))
print("Number of aligned labels:", len(aligned_labels))

Number of subtokens: 27
Number of aligned labels: 27


The 27 input tokens have 27 aligned labels. Only the first subtoken of each word receives its UPOS label; special tokens and subsequent subtokens receive -100 and are excluded from the loss calculation.

### Load Development Data

In [10]:
dev_path = data_dir / "hi_hdtb-ud-dev.conllu"
dev_sentences = read_conllu_sentences(dev_path)

print("Development sentences:", len(dev_sentences))

Development sentences: 1659


The development dataset was loaded successfully, containing 1,659 sentences for model evaluation during training.

### Load Test Data

In [32]:
test_path = data_dir / "hi_hdtb-ud-test.conllu"
test_sentences = read_conllu_sentences(test_path)

print("Test sentences:", len(test_sentences))

Test sentences: 1684


The test dataset was loaded successfully, containing 1,684 sentences for final model evaluation.

### Tokenize and Align Dataset Splits

In [33]:
train_data = tokenize_and_align_labels(train_sample, tokenizer, label2id)

dev_data = tokenize_and_align_labels(dev_sentences, tokenizer, label2id)

test_data = tokenize_and_align_labels(test_sentences, tokenizer, label2id)

print("Training sentences tokenized:", len(train_data["input_ids"]))
print("Development sentences tokenized:", len(dev_data["input_ids"]))
print("Test sentences tokenized:", len(test_data["input_ids"]))

Training sentences tokenized: 2500
Development sentences tokenized: 1659
Test sentences tokenized: 1684


Tokenization and label alignment were completed for 2,500 training sentences, 1,659 development sentences, and 1,684 test sentences.

### Load the Token Classification Model

In [12]:
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(upos_label), id2label=id2label, label2id=label2id)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

print("Model loaded:", model.config.num_labels, "labels on", device)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: 16 labels on mps


The multilingual DistilBERT model loaded with 16 UPOS labels on Apple MPS. The classification layer was newly initialized, so the model needs fine-tuning on the Hindi UPOS training data before making reliable predictions.

### Create Hugging Face Datasets

In [35]:
train_dataset = Dataset.from_dict(train_data)
dev_dataset = Dataset.from_dict(dev_data)
test_dataset = Dataset.from_dict(test_data)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(dev_dataset))
print("Test examples:", len(test_dataset))

Training examples: 2500
Validation examples: 1659
Test examples: 1684


Created Hugging Face datasets containing 2,500 training examples, 1,659 validation examples, and 1,684 test examples.

### Configure Training

In [14]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(output_dir="outputs/upos_baseline",
                                  num_train_epochs=2,
                                 per_device_train_batch_size=8,
                                 per_device_eval_batch_size=8,
                                 eval_strategy="epoch",
                                 save_strategy="no",
                                 logging_steps=50,
                                 report_to="none",
                                 seed=42,
                                 use_cpu=False)

trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=train_dataset,
                  eval_dataset=dev_dataset,
                  data_collator=data_collator)

The model is set to train for 2 epochs with a batch size of 8, evaluate after each epoch on the validation set, and use dynamic token-level padding. Checkpoint saving and reporting integrations are disabled; the random seed is 42.

### Train the Baseline Model

In [15]:
train_result = trainer.train()

/opt/anaconda3/envs/upos_lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.247300,0.190525
2,0.137200,0.169490


The model completed 2 training epochs.Training loss decreased from 0.2473 to 0.1372, while validation loss decreased from 0.1905 to 0.1695, indicating improvement during training.

### Evaluate the Baseline Model

In [16]:
predictions = trainer.predict(dev_dataset)

baseline_metrics = evaluate_token_predictions(predictions.predictions, predictions.label_ids)

baseline_metrics

{'accuracy': 0.9464889596180625,
 'precision': 0.8807843651707319,
 'recall': 0.8756432944709532,
 'f1-score': 0.8780344229201599}

Accuracy: 94.65%, precision: 88.08%, recall: 87.56%, and F1-score: 87.80%.

### Save Baseline Metrics

In [ ]:
output_dir = project_root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / "baseline_metrics.json"

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(baseline_metrics, file, indent=4)

print("Baseline metrics saved.")

Baseline metrics saved.


Baseline evaluation metrics were saved to outputs/baseline_metrics.json.

### Inspect Transformer Layers

In [18]:
print(f"Number of Transformer layers: {model.config.n_layers}")

Number of Transformer layers: 6


The multilingual DistilBERT model contains 6 Transformer layers.

### Freeze the First Three Transformer Layers

In [19]:
frozen_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(upos_label),id2label=id2label, label2id=label2id)

#Freeze the first 3 layers
for layer in frozen_model.distilbert.transformer.layer[:3]:
    for parameter in layer.parameters():
        parameter.requires_grad = False

frozen_model.to(device)

print("Frozen layers: 3 out of 6")
print("Classification head remains trainable:", frozen_model.classifier.weight.requires_grad)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert/distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Frozen layers: 3 out of 6
Classification head remains trainable: True


The first 3 of 6 Transformer layers were frozen, while the classification head remains trainable. This configuration allows fine-tuning the remaining Transformer layers and classification head for Hindi UPOS tagging.

### Calculate Trainable Parameters

In [22]:
total_params = sum(p.numel() for p in frozen_model.parameters())

trainable_params = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)

print(f"Trainable percentage: {trainable_params / total_params * 100:.2f}%")

Trainable percentage: 84.22%


84.22% of the model's parameters remain trainable after freezing the first three Transformer layers.

### Train the Partially Frozen Model

In [23]:
frozen_args = TrainingArguments(output_dir="outputs/upos_partial_freezing",
                                num_train_epochs=2,
                                per_device_train_batch_size=8,
                                per_device_eval_batch_size=8,
                                eval_strategy="epoch",
                                save_strategy="no",
                                logging_steps=50,
                                report_to="none",
                                seed=42)


frozen_trainer = Trainer(model=frozen_model,
                         args=frozen_args,
                         train_dataset=train_dataset,
                         eval_dataset=dev_dataset,
                         data_collator=data_collator)

frozen_result = frozen_trainer.train()

/opt/anaconda3/envs/upos_lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.307200,0.237232
2,0.205800,0.206577


After 2 epochs, training loss decreased from 0.3072 to 0.2058, and validation loss decreased from 0.2372 to 0.2066.

### Evaluate the Partially Frozen Model

In [24]:
predictions = frozen_trainer.predict(dev_dataset)

partial_freezing_metrics = evaluate_token_predictions(predictions.predictions, predictions.label_ids)

partial_freezing_metrics

{'accuracy': 0.9310580010798829,
 'precision': 0.8637622158584141,
 'recall': 0.8487314610833153,
 'f1-score': 0.8551131927002765}

Accuracy: 93.11%, precision: 86.38%, recall: 84.87%, and F1-score: 85.51%. The F1-score is lower than the baseline model's 87.80%.

### Save Partial-Freezing Metrics


In [25]:
metrics_path = project_root / "outputs" /"partial_freezing_metrics.json"

with metrics_path.open("w", encoding='utf-8') as file:
    json.dump(partial_freezing_metrics, file, indent=4)

print("Partial-freezing metrics saved.")

Partial-freezing metrics saved.


Partial-freezing evaluation metrics were saved to outputs/partial_freezing_metrics.json.

### Evaluate the Baseline on the Held-Out Test Set

In [37]:
#Predict the held-out Hindi test set
baseline_predictions = trainer.predict(test_dataset)

baseline_test_metrics = evaluate_token_predictions(baseline_predictions.predictions, baseline_predictions.label_ids)

baseline_test_metrics

/opt/anaconda3/envs/upos_lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'accuracy': 0.9467370441458733,
 'precision': 0.8838408807053276,
 'recall': 0.8802123576416804,
 'f1-score': 0.8818552708692585}

Accuracy: 94.67%, precision: 88.38%, recall: 88.02%, and F1-score: 88.19%.

### Evaluate the Partially Frozen Model on the Held-Out Test Set

In [38]:
frozen_predictions = frozen_trainer.predict(test_dataset)

frozen_test_metrics = evaluate_token_predictions(frozen_predictions.predictions, frozen_predictions.label_ids)

frozen_test_metrics

/opt/anaconda3/envs/upos_lab/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


{'accuracy': 0.9319747092695043,
 'precision': 0.8688985042110169,
 'recall': 0.8563266577274932,
 'f1-score': 0.8617595569144197}

Accuracy: 93.20%, precision: 86.89%, recall: 85.63%, and F1-score: 86.18%. The partially frozen model achieved a lower test F1-score than the baseline model (88.19%).

### Save Test Metrics

In [39]:
test_results = {"baseline_test": baseline_test_metrics,
                "partial_freezing_test": frozen_test_metrics}

test_path = project_root / "outputs" / "test_metrics.json"

with test_path.open("w", encoding='utf-8') as file:
    json.dump(test_results, file, indent=2)

print("Test metrics saved!")

Test metrics saved!


Test metrics for both the baseline and partially frozen models were saved to outputs/test_metrics.json.